# 03. Unsupervised Learning & Cluster-Aware Forecasting (Person B)

This notebook presents the complete unsupervised learning workflow and cluster-aware forecasting experiments for wind and solar energy production in France.

### Objectives:
1. **Daily Profile Matrix Construction**: Reshape hourly time series into daily 24-hour production profiles.
2. **Normalization & Diagnostic Analysis**: Standardize profiles and select optimal cluster count $k$ using Elbow and Silhouette analysis.
3. **Clustering Algorithm Comparison**: Fit and compare **K-Means**, **Gaussian Mixture Models (GMM)**, and **Hierarchical Agglomerative Clustering (Ward)**.
4. **Cluster Profile & Seasonal Interpretation**: Interpret centroids, map daily profiles, and analyze monthly cluster distributions.
5. **PCA Dimensionality Reduction**: Visualize 24-dimensional daily vectors in 2D space.
6. **Cluster-Aware Forecasting Experiments**:
   - **Experiment (a)**: Fit one specialized forecasting model per cluster.
   - **Experiment (b)**: Pass the cluster label as an extra input feature to a global model.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append('../src')
from clustering import (
    build_daily_profile_matrix,
    normalize_profiles,
    evaluate_k,
    fit_clustering_algorithms,
    plot_dendrogram_tree,
    interpret_and_plot_clusters,
    plot_pca_visualization,
    merge_cluster_labels_to_features,
    run_cluster_experiments
)

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load Data & Build Daily Profile Matrix

In [ ]:
cleaned_path = '../data/processed/cleaned.csv'
df_cleaned = pd.read_csv(cleaned_path, index_col='datetime', parse_dates=True)

pivot_raw = build_daily_profile_matrix(df_cleaned)
pivot_scaled = normalize_profiles(pivot_raw)

print(f"Raw Daily Matrix Shape: {pivot_raw.shape} (Days x 24 Hours)")
pivot_raw.head()

## 2. Cluster Count Selection (Elbow & Silhouette Analysis)

In [ ]:
k_metrics = evaluate_k(pivot_scaled, max_k=10, save_plot=True)
k_metrics

## 3. Fit K-Means, GMM, and Hierarchical Models ($k=3$)

In [ ]:
best_k = 3
labels_df, algo_summary = fit_clustering_algorithms(pivot_scaled, k=best_k)
print("Algorithm Comparison (Silhouette Score):")
display(algo_summary)

## 4. Hierarchical Dendrogram

In [ ]:
plot_dendrogram_tree(pivot_scaled)

## 5. Cluster Interpretation & Daily Profiles Visualization

In [ ]:
interpreted_df, cluster_names = interpret_and_plot_clusters(pivot_raw, labels_df, algorithm='kmeans')

## 6. PCA 2D Dimensionality Reduction

In [ ]:
pca_df = plot_pca_visualization(pivot_scaled, labels_df, algorithm='kmeans')

## 7. Merge Cluster Labels & Export Datasets

In [ ]:
train_clustered, test_clustered = merge_cluster_labels_to_features(labels_df)
print(f"Train clustered shape: {train_clustered.shape}")
print(f"Test clustered shape: {test_clustered.shape}")

## 8. Run Cluster-Aware Forecasting Experiments

In [ ]:
exp_results = run_cluster_experiments()
display(exp_results)